<a href="https://colab.research.google.com/github/LASHMANA15/Data-Engineering-and-GenAI-on-cloud/blob/main/Day-4/Day_4_Data_engineering_and_genAI_on_cloud.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pyspark

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("LargeSalesData") \
    .getOrCreate()

df_bronze=spark.read \
    .option("header","true") \
    .option("inferSchema","true") \
    .csv("/content/large_sales_data.csv")
print(f'Bronze table row count: {df_bronze.count()}')
print(f'Bronze table column count: {len(df_bronze.columns)}')

Bronze table row count: 5000
Bronze table column count: 13


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import year,month,to_date,col,round as spark_round
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

In [ ]:
spark = SparkSession.builder \
    .appName('DAY4_BigData_Sales') \
    .config('spark.some.config.option', 'some-value') \
    .getOrCreate()

In [ ]:
df_bronze.select('quantity','unit_price','revenue').describe().show()

+-------+-----------------+------------------+------------------+
|summary|         quantity|        unit_price|           revenue|
+-------+-----------------+------------------+------------------+
|  count|             5000|              5000|              5000|
|   mean|           7.9536|          12496.86|          99169.52|
| stddev|4.275313169878912|14857.384309295603|145972.97195261103|
|    min|                1|               600|               600|
|    max|               15|             45000|            675000|
+-------+-----------------+------------------+------------------+



In [ ]:
df_bronze.write \
    .mode('overwrite') \
    .parquet('sales_bronze.parquet')

In [ ]:
import os
def get_dir_size(path):
  if os.path.isfile(path):
    return os.path.getsize(path)/1024
  total=0
  for dirpath, dirnames, filenames in os.walk(path):
    for f in filenames:
      fp=os.path.join(dirpath,f)
      total+=os.path.getsize(fp)/1024
  return total

csv_size=get_dir_size('large_sales_data.csv')
parquet_size=get_dir_size('sales_bronze.parquet')
reduction=round(csv_size/parquet_size,2)
print(f'CSV file size: {csv_size} KB')
print(f'Parquet file size: {parquet_size} KB')

CSV file size: 529.3125 KB
Parquet file size: 55.09765625 KB


In [ ]:
from pyspark.sql.functions import col
df_bronze = df_bronze.withColumn("calculated_quantity", col("revenue") / col("unit_price"))
df_bronze.select("unit_price", "revenue", "calculated_quantity", "quantity").show(5)
df_bronze.select("revenue").show(12)
df_bronze.filter("revenue>120000")

+----------+-------+-------------------+--------+
|unit_price|revenue|calculated_quantity|quantity|
+----------+-------+-------------------+--------+
|     22000| 264000|               12.0|      12|
|     12000| 120000|               10.0|      10|
|       800|   8000|               10.0|      10|
|     32000| 160000|                5.0|       5|
|      3500|  14000|                4.0|       4|
+----------+-------+-------------------+--------+
only showing top 5 rows
+-------+
|revenue|
+-------+
| 264000|
| 120000|
|   8000|
| 160000|
|  14000|
|  25000|
|   5400|
| 585000|
|  38500|
|  35000|
|   9000|
|  10400|
+-------+
only showing top 12 rows


DataFrame[order_id: int, customer_name: string, product: string, category: string, quantity: int, unit_price: int, revenue: int, order_date: date, city: string, region: string, sales_rep: string, payment_method: string, order_status: string, calculated_quantity: double]

In [ ]:
# Filtering rows where revenue is greater than 100,000
df_high_revenue = df_bronze.filter(col("revenue") > 100000)

# Showing the result
df_high_revenue.select("order_id", "product", "revenue").show(10)

+--------+-------+-------+
|order_id|product|revenue|
+--------+-------+-------+
|    1001|Monitor| 264000|
|    1002|Printer| 120000|
|    1004| Tablet| 160000|
|    1008| Laptop| 585000|
|    1019|Printer| 108000|
|    1021| Laptop| 675000|
|    1041|Printer| 108000|
|    1043|Monitor| 110000|
|    1045| Laptop| 495000|
|    1047|Monitor| 330000|
+--------+-------+-------+
only showing top 10 rows


In [ ]:
df_sliver=df_bronze \
    .dropDuplicates() \
    .dropna(subset=['quantity','unit_price','revenue'])
df_sliver=df_sliver.withColumn("order_date",to_date(col("order_date"),"yyyy-MM-dd")) \
    .withColumn("year",year(col("order_date"))) \
    .withColumn("month",month(col("order_date")))
df_sliver=df_sliver.withColumn("revenue_category",F.when(col('revenue')>40000,'High')\
                                        .when((col('revenue')>10000) & (col('revenue')<=40000),'Medium')\
                                        .otherwise('Low'))
df_sliver.show(5)
print(f'Sliver table row count: {df_sliver.count()}')
print(f'Sliver table column count: {len(df_sliver.columns)}')

+--------+-------------+-------+-----------+--------+----------+-------+----------+---------+------+------------+--------------+------------+-------------------+----+-----+----------------+
|order_id|customer_name|product|   category|quantity|unit_price|revenue|order_date|     city|region|   sales_rep|payment_method|order_status|calculated_quantity|year|month|revenue_category|
+--------+-------------+-------+-----------+--------+----------+-------+----------+---------+------+------------+--------------+------------+-------------------+----+-----+----------------+
|    1019|Kavya Nambiar|Printer|Electronics|       9|     12000| 108000|2023-02-24|Bangalore| South|Deepak Joshi|           UPI|     Shipped|                9.0|2023|    2|            High|
|    1454|  Sneha Reddy|Speaker|Electronics|       3|      4500|  13500|2023-10-16|    Surat|  West| Kavya Reddy|   Net Banking|     Shipped|                3.0|2023|   10|          Medium|
|    1608|  Meera Joshi| Tablet|Electronics|      

In [ ]:
print(f'Bronze Size: {get_dir_size("/content/sales_bronze.parquet")} KB')
print(f'Sliver Size: {get_dir_size("/content/sales_sliver.parquet")} KB')

Bronze Size: 55.09765625 KB
Sliver Size: 0 KB


In [ ]:
df_sliver.write \
    .mode('overwrite') \
    .parquet('sales_sliver.parquet')
df_verify=spark.read.parquet('/content/sales_sliver.parquet')
df_verify.show(5)
df_verify.printSchema()

+--------+-------------+-------+-----------+--------+----------+-------+----------+---------+------+------------+--------------+------------+-------------------+----+-----+----------------+
|order_id|customer_name|product|   category|quantity|unit_price|revenue|order_date|     city|region|   sales_rep|payment_method|order_status|calculated_quantity|year|month|revenue_category|
+--------+-------------+-------+-----------+--------+----------+-------+----------+---------+------+------------+--------------+------------+-------------------+----+-----+----------------+
|    1019|Kavya Nambiar|Printer|Electronics|       9|     12000| 108000|2023-02-24|Bangalore| South|Deepak Joshi|           UPI|     Shipped|                9.0|2023|    2|            High|
|    1454|  Sneha Reddy|Speaker|Electronics|       3|      4500|  13500|2023-10-16|    Surat|  West| Kavya Reddy|   Net Banking|     Shipped|                3.0|2023|   10|          Medium|
|    1608|  Meera Joshi| Tablet|Electronics|      

In [ ]:
print('Top 5 products by total revenue (Ascending order):')
df_sliver.groupBy('product').agg(F.sum('revenue').alias('total_revenue')).orderBy(F.asc('total_revenue')).limit(5).show()

print('\nTop 5 products by total revenue (Descending order):')
df_sliver.groupBy('product').agg(F.sum('revenue').alias('total_revenue')).orderBy(F.desc('total_revenue')).limit(5).show()

print('\nTop 10 products by total revenue (Descending order):')
df_sliver.groupBy('product').agg(F.sum('revenue').alias('total_revenue')).orderBy(F.desc('total_revenue')).limit(10).show()

Top 5 products by total revenue (Ascending order):
+----------+-------------+
|   product|total_revenue|
+----------+-------------+
|   USB Hub|      2447400|
|     Mouse|      3207200|
|  Keyboard|      4878000|
|    Webcam|     10982500|
|Headphones|     13541500|
+----------+-------------+


Top 5 products by total revenue (Descending order):
+-------+-------------+
|product|total_revenue|
+-------+-------------+
| Laptop|    182700000|
| Tablet|    135104000|
|Monitor|     82126000|
|Printer|     44544000|
|Speaker|     16317000|
+-------+-------------+


Top 10 products by total revenue (Descending order):
+----------+-------------+
|   product|total_revenue|
+----------+-------------+
|    Laptop|    182700000|
|    Tablet|    135104000|
|   Monitor|     82126000|
|   Printer|     44544000|
|   Speaker|     16317000|
|Headphones|     13541500|
|    Webcam|     10982500|
|  Keyboard|      4878000|
|     Mouse|      3207200|
|   USB Hub|      2447400|
+----------+-------------+

